# broadcast-initial-weights — worked example 1: Sync every parameter from rank 0 with a fake dist backend

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-initial-weights`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When data-parallel training starts, each rank builds its own model and gets its OWN random init. Before the first forward pass, you must overwrite every replica's `param.data` with rank 0's values via `dist.broadcast(p.data, src=0)`. Because `broadcast` mutates the tensor storage in-place, looping over `model.parameters()` is enough — no `load_state_dict` needed.

## Worked solution

**Goal.** Show that after a broadcast loop, a divergent replica ends up identical to rank 0.

1. **Stub the collective.** We can't spawn real processes in a notebook, so we build a tiny `FakeDist` that holds rank 0's canonical tensors. Its `broadcast(tensor, src)` copies the source-rank value into `tensor` in-place — exactly the contract of the real `dist.broadcast`.
2. **Make ranks diverge.** We fill rank 1's `Linear` weight with `2.0` and rank 0's with `1.0`. If we forgot to sync, rank 1 would train on the wrong weights.
3. **The sync loop.** `for p in model.parameters(): fake.broadcast(p.data, src=0)`. Iterating `parameters()` is deterministic and identical across ranks because the model graph is identical, so tensor i on every rank maps to tensor i on rank 0.
4. **Why in-place matters.** `broadcast` writes into the existing storage, so the very same `nn.Parameter` object now holds rank 0's numbers — the optimizer and forward pass automatically see the synced values.
5. **Check.** Rank 1's weight is now all `1.0`, matching rank 0.

In [ ]:
class FakeDist:
    """Stand-in for torch.distributed that mimics broadcast's in-place copy."""
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors  # what src=0 holds
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        # src rank's value wins; copy_ mutates in place like the real API.
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Linear(2, 2, bias=False)
    with t.no_grad():
        m.weight.fill_(float(rank + 1))
    return m

def broadcast_params(model, fake):
    fake.reset()
    for p in model.parameters():
        fake.broadcast(p.data, src=0)

rank0 = build_model(0)
fake = FakeDist([p.data.clone() for p in rank0.parameters()])

rank1 = build_model(1)
print('rank1 before:', rank1.weight.detach().flatten().tolist())
broadcast_params(rank1, fake)
print('rank1 after: ', rank1.weight.detach().flatten().tolist())
print('matches rank0:', t.equal(rank1.weight.data, rank0.weight.data))